# Конкатенация mean-pooling с другими пулингами. Базовый энкодер.

## Цель эксперимента

Сравнить метрики baseline и полученного решения.

## Детали
### Методы
1. Autoencoder. Учится хорошо восстанавливать нормальные данные.

Выбор метода обусловлен его высокой вычислительной эффективностью при большом объеме расчетов и высокой размерности векторов. В предыдущих экспериментах Autoencoder продемонстрировал наилучшие показатели, но он недетерминирован на MPS.

**Важно: рассчеты были проведены на MPS GPU, который недетерминирован (нельзя задать seed для воспроизводимости). Поэтому значения метрик могут варьироваться от эксперимента к эксперименту.**

### Энкодеры

1. all-mpnet-base-v2

In [1]:
ENCODERS = [
    "all-mpnet-base-v2",
]

ENCODE_BATCH_SIZE = 32

AE_EPOCH_N = 5
AE_BATCH_SIZE = 8192

CACHE_DIR = "../emb_cache"

device = "mps"

In [2]:
import numpy as np
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    balanced_accuracy_score,
    roc_curve
)
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
from sentence_transformers import SentenceTransformer
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import re
import random
from tqdm import tqdm
from collections import defaultdict
import hashlib

import nltk
nltk.download('punkt')

[nltk_data] Downloading package punkt to /Users/artfultom/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [3]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

In [4]:
import os
from dotenv import load_dotenv

load_dotenv()
os.environ["TOKENIZERS_PARALLELISM"] = "false"

## Загрузка датасетов
- TRAIN: Живые эссе с Ivy Panda, 90805 штук.
- TEST: Живые эссе с Ivy Panda + сгенерированные (Mistral 7B, Llama 3 13B, ChatGPT, DeepSeek). Пропорции 50/50. Суммарно 74976 штук.

In [5]:
filenames = [
    "../datasets/mistral_essays_1.csv",
    "../datasets/mistral_essays_2.csv",
    "../datasets/mistral_essays_3.csv",
    "../datasets/mistral_essays_4.csv",
    "../datasets/mistral_essays_5.csv",
    "../datasets/llama_essays_1.csv",
    "../datasets/llama_essays_2.csv",
    "../datasets/llama_essays_3.csv",
    "../datasets/llama_essays_4.csv",
    "../datasets/llama_essays_5.csv",
    "../datasets/deepseek_essays_1.csv",
    "../datasets/deepseek_essays_2.csv",
    "../datasets/deepseek_essays_3.csv",
    "../datasets/deepseek_essays_4.csv",
    "../datasets/deepseek_essays_5.csv",
    "../datasets/chatgpt_essays_1.csv",
    "../datasets/chatgpt_essays_2.csv",
    "../datasets/chatgpt_essays_3.csv",
    "../datasets/chatgpt_essays_4.csv",
    "../datasets/chatgpt_essays_5.csv",
]
df_ai = pd.concat([pd.read_csv(f) for f in filenames], ignore_index=True).assign(label=1)[["text", "label"]]#.sample(n=200, random_state=SEED)

df_human = pd.read_csv("../datasets/ivy_panda_essays.csv").assign(label=0)[["text", "label"]].reset_index(drop=True)#.sample(n=400, random_state=SEED)
df_train, df_human_test = train_test_split(df_human, test_size=len(df_ai), random_state=SEED)

df_all = pd.concat([df_ai, df_human_test], ignore_index=True).sample(frac=1, random_state=SEED)

In [6]:
print(f"Загружено {len(df_train)} записей Ivy Panda (живые, train)")
print(f"Загружено {len(df_all)} записей (синтетика + живые, test)")

Загружено 90805 записей Ivy Panda (живые, train)
Загружено 74976 записей (синтетика + живые, test)


### Необходимые для моделей классы

In [7]:
class AE(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.enc = nn.Sequential(
            nn.Linear(dim, 128),
            nn.ReLU(),
            nn.Linear(128, 32),
            nn.ReLU(),
        )
        self.dec = nn.Sequential(
            nn.Linear(32, 128),
            nn.ReLU(),
            nn.Linear(128, dim),
        )
    def forward(self, x):
        return self.dec(self.enc(x))

In [8]:
def normalize_f(data):
    data = torch.from_numpy(data).float()
    data = F.normalize(data, p=2, dim=0)
    return data.numpy()

def _compute_hash(data, model_name):
    h = hashlib.sha256()
    h.update(model_name.encode("utf-8"))

    for text in data:
        h.update(text.encode("utf-8"))

    return h.hexdigest()[:32]

def get_embeddings(model, data, model_name, cache_dir=CACHE_DIR):
    safe_model_name = model_name.replace("/", "__")
    
    os.makedirs(cache_dir, exist_ok=True)

    cache_key = _compute_hash(data, model_name)
    cache_path = os.path.join(cache_dir, f"{safe_model_name}_{cache_key}_sentences.npy")
    cache_counts_path = os.path.join(cache_dir, f"{safe_model_name}_{cache_key}_counts.npy")

    if os.path.exists(cache_path) and os.path.exists(cache_counts_path):
        print(f"Loading embeddings from cache for {model_name}...")
        all_embeddings = np.load(cache_path)
        sentence_counts = np.load(cache_counts_path)
    else:
        print(f"Computing embeddings for {model_name}...")
    
        all_sentences = []
        sentence_counts = []
        for text in tqdm(data):
            text = re.sub(r"\n+", ". ", text)
            sentences = nltk.sent_tokenize(text)
    
            if "e5" in model_name:
                sentences = [f"passage: {s}" for s in sentences]
    
            all_sentences.extend(sentences)
            sentence_counts.append(len(sentences))
    
        all_embeddings = model.encode(
            all_sentences,
            batch_size=ENCODE_BATCH_SIZE,
            normalize_embeddings=True,
            show_progress_bar=True,
            device=DEVICE,
        )

        np.save(cache_path, all_embeddings)
        np.save(cache_counts_path, sentence_counts)

    mean_vals = []
    mean_diff_vals = []
    
    idx = 0
    for count in tqdm(sentence_counts):
        embeddings = all_embeddings[idx: idx + count]
        mean_vals.append(np.mean(embeddings, axis=0))
        
        diffs = embeddings[1:] - embeddings[:-1]
        mean_diff_val = np.mean(diffs, axis=0)
        mean_diff_val = normalize_f(mean_diff_val)
        mean_diff_vals.append(mean_diff_val)
        
        idx += count

    result = (
        np.array(mean_vals, dtype=np.float32), 
        np.array(mean_diff_vals, dtype=np.float32),
    )

    return result

In [9]:
def ae_score(X_train, X_test):
    d = X_train.shape[1]
    
    ae = AE(d).to(device)
    ae.train()
    
    opt = torch.optim.Adam(ae.parameters(), lr=1e-3)
    loss_fn = nn.MSELoss()

    Xtr = torch.tensor(X_train, dtype=torch.float32)
    dataset = torch.utils.data.TensorDataset(Xtr)
    loader = torch.utils.data.DataLoader(
        dataset,
        batch_size=AE_BATCH_SIZE,
        shuffle=True,
        drop_last=False
    )

    for epoch in range(AE_EPOCH_N):
        for (batch,) in loader:
            batch = batch.to(device)

            out = ae(batch)
            loss = loss_fn(out, batch)

            opt.zero_grad()
            loss.backward()
            opt.step()

    ae.eval()
    Xe = torch.tensor(X_test, dtype=torch.float32).to(device)

    with torch.no_grad():
        recon = ae(Xe).cpu().numpy()

    return np.mean((X_test - recon) ** 2, axis=1)

In [10]:
def fpr_at_tpr(y_true, scores, target_tpr=0.95):
    fpr, tpr, thresholds = roc_curve(y_true, scores)
    idx = np.where(tpr >= target_tpr)[0]
    if len(idx) == 0:
        return np.nan
        
    return fpr[idx[0]]

def evaluate(y_true, scores):
    fpr, tpr, thresholds = roc_curve(y_true, scores)
    optimal_idx = (tpr - fpr).argmax()
    optimal_threshold = thresholds[optimal_idx]

    y_pred = (scores >= optimal_threshold).astype(int)

    print(f"ROC AUC:\t {roc_auc_score(y_true, scores):.4f}")
    print(f"PR AUC:\t\t {average_precision_score(y_true, scores):.4f}")
    
    print(f"Accuracy:\t {accuracy_score(y_true, y_pred):.4f}")
    
    print(f"Precision:\t {precision_score(y_true, y_pred):.4f}")
    print(f"Recall:\t\t {recall_score(y_true, y_pred):.4f}")
    print(f"F1:\t\t {f1_score(y_true, y_pred):.4f}")

    fpr95 = fpr_at_tpr(y_true, scores, 0.95)
    fpr99 = fpr_at_tpr(y_true, scores, 0.99)

    print(f"FPR@95TPR:\t {fpr95:.4f}")
    print(f"FPR@99TPR:\t {fpr99:.4f}")

    print(f"Best threshold:\t {optimal_threshold:.4f}")
    print("-" * 50)


for model_name in ENCODERS:
    model = SentenceTransformer(model_name)

    train_embeddings = get_embeddings(model, df_train['text'].tolist(), model_name)
    test_embeddings = get_embeddings(model, df_all['text'].tolist(), model_name)
        
    y_test = df_all['label'].values.astype(int)
            
    X_train = np.array(train_embeddings[0])
    X_test = np.array(test_embeddings[0])

    print(f"\n=== {model_name} | baseline embeddings ===")
    ae_scores = ae_score(X_train, X_test)
    evaluate(y_test, ae_scores)

    X_train = np.concatenate([X_train, np.array(train_embeddings[1])], axis=1)
    X_test = np.concatenate([X_test, np.array(test_embeddings[1])], axis=1)

    print(f"\n=== {model_name} | extended embeddings ===")
    ae_scores = ae_score(X_train, X_test)
    evaluate(y_test, ae_scores)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading embeddings from cache for all-mpnet-base-v2...


100%|██████████████████████████████████| 90805/90805 [00:06<00:00, 13190.13it/s]


Loading embeddings from cache for all-mpnet-base-v2...


100%|██████████████████████████████████| 74976/74976 [00:02<00:00, 31564.10it/s]



=== all-mpnet-base-v2 | baseline embeddings ===
ROC AUC:	 0.8632
PR AUC:		 0.8546
Accuracy:	 0.7843
Precision:	 0.7731
Recall:		 0.8050
F1:		 0.7887
FPR@95TPR:	 0.5241
FPR@99TPR:	 0.7738
Best threshold:	 0.0003
--------------------------------------------------

=== all-mpnet-base-v2 | extended embeddings ===
ROC AUC:	 0.9207
PR AUC:		 0.9078
Accuracy:	 0.8474
Precision:	 0.8195
Recall:		 0.8910
F1:		 0.8538
FPR@95TPR:	 0.2787
FPR@99TPR:	 0.5988
Best threshold:	 0.0008
--------------------------------------------------


## Выводы

Наблюдается устойчивый рост всех метрик.